# Notebook 09 — Technical Indicators + Validity Tracking

**Input** → `data/investable_universe.parquet` (140 rows, 5 companies)

**Output** → `data/indicators.parquet` (140 rows, 31+ columns including validity status)

**Indicators computed per company on consecutive Cours values**

| Family | Indicator | Min obs (strict) | Min obs (reliable) | Validity tracked |
|---|---|---|---|---|
| Trend | SMA_20 | 20 | 20 | ✓ |
| Trend | SMA_50 | 50 | 50 | ✓ |
| Trend | EMA_20 | 1 | 20 (3× span) | ✓ |
| Momentum | RSI_14 | 15 | 15 | ✓ |
| Momentum | MACD / Signal / Histogram | 26 | 35 | ✓ |
| Volume | RVOL | 1 | 20 | ✓ |
| Volume | VWAP | 1 | 1 (cumulative) | ✓ |
| Volatility | HV_20 | 21 | 21 | ✓ |

**Important distinction:**
- **Min obs (strict)**: Technical minimum for formula to execute (may produce meaningless values)
- **Min obs (reliable)**: Minimum for statistically meaningful result

For example, EMA_20 uses `min_periods=1` (degrades gracefully), but results aren't reliable until ~20 observations.

**NEW in this version: Per-indicator validity status**

Each indicator now has a `Valid_{indicator}` column:
- `VALID` → indicator computed successfully (non-NaN)
- `INSUFFICIENT_DATA` → indicator is NaN (not enough history or missing price data)

This addresses the feedback: "production with a full history all indicators fire" is NOT guaranteed.
A company can have sufficient overall history but still have gaps in specific periods.
The validity status makes this explicit row-by-row.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
def _find_root(start):
    for c in [start, start.parent, start.parent.parent]:
        if (c / 'src').exists() and (c / 'data').exists():
            return c.resolve()
    raise RuntimeError(f'Cannot locate project root from {start}')
ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.validation import load_unified_dataset, save_unified_dataset
from config.methodology import INDICATOR_PARAMS as P, INDICATOR_MIN_OBS as MOBS
pd.set_option('display.float_format', '{:.4f}'.format)
print(f'ROOT: {ROOT}')

ROOT: /home/yass/Desktop/DSS_CMR


## Step 1 — Load investable universe

In [2]:
inv, _ = load_unified_dataset(str(ROOT / 'data' / 'investable_universe.parquet'))
print(f'Shape: {inv.shape} | companies: {inv["CODE_ISIN"].nunique()}')
print(inv[['CODE_ISIN','Company']].drop_duplicates().to_string(index=False))
print(f'\nCours coverage per company:')
for isin, g in inv.groupby('CODE_ISIN'):
    n = g['Cours'].notna().sum()
    print(f'  {g["Company"].iloc[0]:25s} {n:2d}/{len(g)} sessions with Cours')

Shape: (140, 11) | companies: 5
   CODE_ISIN            Company
MA0000010936 ALUMINIUM DU MAROC
MA0000010944               AGMA
MA0000010951       AFRIQUIA GAZ
MA0000011819          ALLIANCES
MA0000012296               AFMA

Cours coverage per company:
  ALUMINIUM DU MAROC        14/28 sessions with Cours
  AGMA                      14/28 sessions with Cours
  AFRIQUIA GAZ              14/28 sessions with Cours
  ALLIANCES                 14/28 sessions with Cours
  AFMA                      14/28 sessions with Cours


## Step 2 — Indicator computation functions

In [3]:
def _rsi_wilder(prices_clean, period):
    """Wilder RSI on clean (no-NaN) price series."""
    n     = len(prices_clean)
    delta = prices_clean.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    ag    = pd.Series(np.nan, index=prices_clean.index)
    al    = pd.Series(np.nan, index=prices_clean.index)
    if n >= period:
        ag.iloc[period-1] = gain.iloc[:period].mean()
        al.iloc[period-1] = loss.iloc[:period].mean()
        for i in range(period, n):
            ag.iloc[i] = (ag.iloc[i-1]*(period-1) + gain.iloc[i]) / period
            al.iloc[i] = (al.iloc[i-1]*(period-1) + loss.iloc[i]) / period
    rs = ag / al.replace(0, np.nan)
    return 100 - 100/(1+rs)

def compute_indicators(group_df, P, MOBS):
    g = group_df.sort_values('Date').copy()
    IND = ['SMA_20','SMA_50','EMA_20','RSI_14','MACD','MACD_Signal','MACD_Histogram','RVOL','VWAP','HV_20']
    for c in IND: g[c] = np.nan

    mask = g['Cours'].notna()
    if not mask.any(): return g
    pr = g.loc[mask,'Cours']
    n  = len(pr)

    # Trend — SMA (strict: NaN if history too short)
    if n >= MOBS['SMA_20']:
        g.loc[mask,'SMA_20'] = pr.rolling(P['sma_short'], min_periods=P['sma_short']).mean().values
    if n >= MOBS['SMA_50']:
        g.loc[mask,'SMA_50'] = pr.rolling(P['sma_long'],  min_periods=P['sma_long']).mean().values

    # Trend — EMA (min_periods=1: produces value from obs 1 onward)
    g.loc[mask,'EMA_20'] = pr.ewm(span=P['ema_short'], adjust=False, min_periods=1).mean().values

    # Momentum — RSI (Wilder, valid only on obs >= period)
    rsi = _rsi_wilder(pr.reset_index(drop=True), P['rsi_period'])
    g.loc[mask,'RSI_14'] = rsi.values

    # Momentum — MACD (strict: NaN if history < slow EMA window)
    ema_f = pr.ewm(span=P['macd_fast'],   adjust=False, min_periods=1).mean()
    ema_s = pr.ewm(span=P['macd_slow'],   adjust=False, min_periods=1).mean()
    ml    = ema_f - ema_s
    sig   = ml.ewm(span=P['macd_signal'], adjust=False, min_periods=1).mean()
    if n >= MOBS['MACD']:
        g.loc[mask,'MACD']           = ml.values
        g.loc[mask,'MACD_Signal']    = sig.values
        g.loc[mask,'MACD_Histogram'] = (ml-sig).values

    # Volume — RVOL
    vm = g['Volume MC'].notna()
    if vm.any():
        vs    = g.loc[vm,'Volume MC']
        avg_v = vs.rolling(P['rvol_window'], min_periods=1).mean()
        g.loc[vm,'RVOL'] = (vs / avg_v.replace(0,np.nan)).values

    # Volume — VWAP cumulative
    pv_price = g['Cours'].fillna(((g['Bid'].fillna(0)+g['Ask'].fillna(0))/2).replace(0,np.nan))
    both = pv_price.notna() & g['Volume MC'].notna()
    if both.any():
        pv      = (pv_price * g['Volume MC']).where(both)
        cum_pv  = pv.cumsum()
        cum_vol = g['Volume MC'].where(both,0).cumsum()
        g['VWAP'] = (cum_pv / cum_vol.replace(0,np.nan)).where(cum_vol>0)

    # Volatility — HV_20 annualised
    if n >= P['hv_window']+1:
        lr = np.log(pr / pr.shift(1))
        hv = lr.rolling(P['hv_window'], min_periods=P['hv_window']).std() * np.sqrt(P['hv_annualise'])
        g.loc[mask,'HV_20'] = hv.values

    return g

print('✓ Indicator functions defined')

✓ Indicator functions defined


## Step 3 — Compute indicators

In [4]:
parts = [compute_indicators(grp, P, MOBS) for _, grp in inv.groupby('CODE_ISIN')]
df_ind = pd.concat(parts).sort_values(['CODE_ISIN','Date']).reset_index(drop=True)
print(f'Output shape: {df_ind.shape}')
IND_COLS = ['SMA_20','SMA_50','EMA_20','RSI_14','MACD','MACD_Signal','MACD_Histogram','RVOL','VWAP','HV_20']
print(f'\nIndicator coverage (non-null rows):')
for c in IND_COLS:
    nn   = df_ind[c].notna().sum()
    note = '' if nn > 0 else '  ← sample too short (valid in production with more history)'
    print(f'  {c:20s}: {nn:4d}/{len(df_ind)} ({nn/len(df_ind)*100:.0f}%){note}')

Output shape: (140, 21)

Indicator coverage (non-null rows):
  SMA_20              :    0/140 (0%)  ← sample too short (valid in production with more history)
  SMA_50              :    0/140 (0%)  ← sample too short (valid in production with more history)
  EMA_20              :   70/140 (50%)
  RSI_14              :    5/140 (4%)
  MACD                :    0/140 (0%)  ← sample too short (valid in production with more history)
  MACD_Signal         :    0/140 (0%)  ← sample too short (valid in production with more history)
  MACD_Histogram      :    0/140 (0%)  ← sample too short (valid in production with more history)
  RVOL                :   30/140 (21%)
  VWAP                :   30/140 (21%)
  HV_20               :    0/140 (0%)  ← sample too short (valid in production with more history)


## Step 4 — **NEW: Add validity status per indicator**

In [5]:
def indicator_validity(row, indicator_col):
    """Return VALID if indicator is non-NaN, INSUFFICIENT_DATA otherwise."""
    return 'VALID' if pd.notna(row.get(indicator_col, np.nan)) else 'INSUFFICIENT_DATA'

for ind in IND_COLS:
    df_ind[f'Valid_{ind}'] = df_ind.apply(lambda r: indicator_validity(r, ind), axis=1)

print('Validity status columns added:')
print([c for c in df_ind.columns if c.startswith('Valid_')])

Validity status columns added:
['Valid_SMA_20', 'Valid_SMA_50', 'Valid_EMA_20', 'Valid_RSI_14', 'Valid_MACD', 'Valid_MACD_Signal', 'Valid_MACD_Histogram', 'Valid_RVOL', 'Valid_VWAP', 'Valid_HV_20']


## Step 5 — Validity summary per company

In [6]:
validity_rows = []
for isin, grp in df_ind.groupby('CODE_ISIN'):
    row = {'CODE_ISIN': isin, 'Company': grp['Company'].iloc[0]}
    for ind in IND_COLS:
        valid = (grp[f'Valid_{ind}'] == 'VALID').sum()
        total = len(grp)
        row[ind] = f'{valid}/{total}'
    validity_rows.append(row)

validity_df = pd.DataFrame(validity_rows)
print('Indicator validity summary (VALID rows / total rows per company):')
print(validity_df.to_string(index=False))
print()
print('Interpretation:')
print('  EMA_20  14/28 → computed on all 14 sessions with Cours (50% of total rows)')
print('  RSI_14   1/28 → only last Cours row has enough history for Wilder RSI')
print('  SMA_20/50, MACD, HV_20  0/28 → need more obs than our 14-Cours sample')
print()
print('In production with 6+ months of history, most indicators will be VALID.')

Indicator validity summary (VALID rows / total rows per company):
   CODE_ISIN            Company SMA_20 SMA_50 EMA_20 RSI_14 MACD MACD_Signal MACD_Histogram  RVOL  VWAP HV_20
MA0000010936 ALUMINIUM DU MAROC   0/28   0/28  14/28   1/28 0/28        0/28           0/28  5/28  5/28  0/28
MA0000010944               AGMA   0/28   0/28  14/28   1/28 0/28        0/28           0/28  1/28  1/28  0/28
MA0000010951       AFRIQUIA GAZ   0/28   0/28  14/28   1/28 0/28        0/28           0/28  5/28  5/28  0/28
MA0000011819          ALLIANCES   0/28   0/28  14/28   1/28 0/28        0/28           0/28 14/28 14/28  0/28
MA0000012296               AFMA   0/28   0/28  14/28   1/28 0/28        0/28           0/28  5/28  5/28  0/28

Interpretation:
  EMA_20  14/28 → computed on all 14 sessions with Cours (50% of total rows)
  RSI_14   1/28 → only last Cours row has enough history for Wilder RSI
  SMA_20/50, MACD, HV_20  0/28 → need more obs than our 14-Cours sample

In production with 6+ months of his

## Step 6 — Sample: one company with validity flags

In [7]:
print('ALLIANCES — last 8 Cours rows with validity status:')
a = df_ind[df_ind['CODE_ISIN']=='MA0000011819']
cols_show = ['Date','Cours','EMA_20','Valid_EMA_20','RSI_14','Valid_RSI_14','RVOL','Valid_RVOL']
print(a[a['Cours'].notna()][cols_show].tail(8).to_string(index=False))

ALLIANCES — last 8 Cours rows with validity status:
      Date   Cours  EMA_20 Valid_EMA_20  RSI_14      Valid_RSI_14   RVOL Valid_RVOL
2019-01-09 78.4000 82.5958        VALID     NaN INSUFFICIENT_DATA 0.1207      VALID
2019-01-10 77.5000 82.1105        VALID     NaN INSUFFICIENT_DATA 0.2495      VALID
2019-01-14 78.0000 81.7190        VALID     NaN INSUFFICIENT_DATA 0.0470      VALID
2019-01-15 79.4900 81.5067        VALID     NaN INSUFFICIENT_DATA 0.1429      VALID
2019-01-16 77.8100 81.1547        VALID     NaN INSUFFICIENT_DATA 0.0816      VALID
2019-01-17 78.0000 80.8542        VALID     NaN INSUFFICIENT_DATA 0.4469      VALID
2019-01-18 77.7000 80.5538        VALID     NaN INSUFFICIENT_DATA 0.3197      VALID
2019-01-21 77.1000 80.2249        VALID 29.2542             VALID 0.5424      VALID


## Step 7 — Save to Parquet

In [8]:
out = str(ROOT/'data'/'indicators.parquet')
rep = save_unified_dataset(df_ind, out)
chk, _ = load_unified_dataset(out)
assert chk.shape == df_ind.shape
print(f'✓ data/indicators.parquet  {rep["rows"]} rows  {rep["file_size_mb"]:.3f} MB')
print(f'  Total columns: {len(df_ind.columns)} (indicators + validity status)')
print(f'  Validity columns: {[c for c in df_ind.columns if c.startswith("Valid_")]}')
print(f'  Round-trip ✓')

✓ data/indicators.parquet  140 rows  0.020 MB
  Total columns: 31 (indicators + validity status)
  Validity columns: ['Valid_SMA_20', 'Valid_SMA_50', 'Valid_EMA_20', 'Valid_RSI_14', 'Valid_MACD', 'Valid_MACD_Signal', 'Valid_MACD_Histogram', 'Valid_RVOL', 'Valid_VWAP', 'Valid_HV_20']
  Round-trip ✓


## Summary

In [9]:
print('NOTEBOOK 09 — TECHNICAL INDICATORS + VALIDITY'.center(70,'='))
print(f'  Input  : {inv.shape}  (investable_universe.parquet)')
print(f'  Output : {df_ind.shape}  (indicators.parquet)')
print(f'  Indicator columns: {IND_COLS}')
print(f'  Validity columns : {[c for c in df_ind.columns if c.startswith("Valid_")]}')
print()
print('  Sample-data caveat:')
print('    Only 14 consecutive Cours obs per company → many indicators NaN.')
print('    Validity status makes this explicit per row.')
print('    In production (6+ months), most indicators will be VALID.')
print()
print('  Key improvement from feedback:')
print('    "Production with full history all indicators fire" is NOT automatic.')
print('    Validity status tracks which indicators are computable on each row.')
print(f'  → Next: Notebook 10 — Business Rules (use validity in Confidence)')
print('='*70)

============NOTEBOOK 09 — TECHNICAL INDICATORS + VALIDITY=============
  Input  : (140, 11)  (investable_universe.parquet)
  Output : (140, 31)  (indicators.parquet)
  Indicator columns: ['SMA_20', 'SMA_50', 'EMA_20', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'RVOL', 'VWAP', 'HV_20']
  Validity columns : ['Valid_SMA_20', 'Valid_SMA_50', 'Valid_EMA_20', 'Valid_RSI_14', 'Valid_MACD', 'Valid_MACD_Signal', 'Valid_MACD_Histogram', 'Valid_RVOL', 'Valid_VWAP', 'Valid_HV_20']

  Sample-data caveat:
    Only 14 consecutive Cours obs per company → many indicators NaN.
    Validity status makes this explicit per row.
    In production (6+ months), most indicators will be VALID.

  Key improvement from feedback:
    "Production with full history all indicators fire" is NOT automatic.
    Validity status tracks which indicators are computable on each row.
  → Next: Notebook 10 — Business Rules (use validity in Confidence)
